# 01 — Station and Study Period Selection

**What this notebook does:**  
Loads raw hourly PM2.5 data downloaded from UK-AIR, computes the annual data capture rate for each monitoring station, and selects the stations and study period that meet the AURN 85% minimum data quality threshold.

**Why this step matters:**  
Before any modelling can begin, we need to know which stations have enough valid data to be usable. Stations with large amounts of missing data would confound our controlled missingness experiments — we cannot distinguish natural gaps from the artificial ones we introduce ourselves.

**Output:**  
This notebook saves the selected PM2.5 and modelled temperature panel for the final study period and stations.

Main processed outputs:
- `data/processed/selected_pm25_temperature_panel.csv`
- `data/processed/selected_pm25_temperature_panel_for_eda.csv`
- `data/processed/selected_pm25_wide_panel.csv`
- `data/processed/selected_temperature_wide_panel.csv`
- `data/processed/selected_station_metadata.csv`.

## Cell 1 — Load libraries

We only need four standard libraries for this notebook. Nothing unusual here.

In [1]:
# pandas  — for building and manipulating data tables
# numpy   — for handling missing values (NaN)
# csv     — for reading the UK-AIR files line by line (they have a non-standard header)
# warnings — we suppress harmless pandas deprecation warnings to keep output clean

import pandas as pd
import numpy as np
import csv
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')


Libraries loaded OK


In [2]:
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Processed output folder ready: {PROCESSED_DIR}")

Processed output folder ready: ../data/processed


## Cell 2 — Define file paths

The raw data was downloaded from UK-AIR in three separate files because the download tool has a size limit. We define the paths here in one place so they are easy to update if the files are moved.

In [3]:
# The '../' moves one folder up from notebooks/ back to Codes/,
# then into data/raw/Greater_London/ where the downloaded files live.

FILES = [
    '../data/raw/Greater_London/2017-2020.csv',
    '../data/raw/Greater_London/2021-2023.csv',
    '../data/raw/Greater_London/2024-2025.csv',
]

print('Files defined:')
for f in FILES:
    print(f'  {f}')


Files defined:
  ../data/raw/Greater_London/2017-2020.csv
  ../data/raw/Greater_London/2021-2023.csv
  ../data/raw/Greater_London/2024-2025.csv


## Cell 3 — Inspect the raw file structure

**Why we do this before writing any parser:**  
UK-AIR CSV files are not standard CSV tables. Python rows 0-9 contain metadata such as file information, station names, coordinates, site types and local authority information. Python row 10 contains the column headers, and the first actual hourly data row starts at Python row index 11.

This is why the parser cannot use a normal `pd.read_csv()` call directly. The raw structure must be inspected first so that the correct rows and columns are used when extracting station names, PM2.5 values and status flags.

In [4]:
# Open the first file and read every row as a plain list of strings.
# We are NOT parsing yet — just looking at the raw UK-AIR structure.
with open(FILES[0], "r", encoding="utf-8", errors="replace") as f:
    raw_rows = list(csv.reader(f))

print(f"Total rows in file: {len(raw_rows)}")
print(f"Total columns in row 0: {len(raw_rows[0])}")
print(f"Total columns in header row 10: {len(raw_rows[10])}")
print()

# Print the first 13 rows showing the first 12 columns.
# Showing more than 4 columns is useful because the file now contains
# both PM2.5 and Modelled Temperature columns.
print("First 13 rows (first 12 columns only):")
print("-" * 80)

for i, row in enumerate(raw_rows[:13]):
    print(f"Row {i:02d}: {row[:12]}")

print()
print("Target variables found in raw header row 10:")
print("-" * 80)

target_raw_variables = [
    "PM2.5 particulate matter (Hourly measured)",
    "Modelled Temperature"
]

for col_index, col_name in enumerate(raw_rows[10]):
    if col_name.strip() in target_raw_variables:
        station_name = raw_rows[3][col_index].strip()

        # In multi-variable UK-AIR files, the station name may be blank
        # for later variables, so look backwards to find the latest station name.
        if station_name == "":
            for previous_col in range(col_index - 1, 1, -1):
                previous_station = raw_rows[3][previous_col].strip()
                if previous_station != "":
                    station_name = previous_station
                    break

        status_col_index = col_index + 1

        print(f"Column {col_index}: {col_name}")
        print(f"  Station/status group: {station_name}")
        print(f"  Status column index : {status_col_index}")
        print()

Total rows in file: 35079
Total columns in row 0: 1
Total columns in header row 10: 58

First 13 rows (first 12 columns only):
--------------------------------------------------------------------------------
Row 00: ['Hourly measurement data supplied by UK-air on 22/06/2026']
Row 01: ['All Data GMT hour ending']
Row 02: ['Status: V=Verified P=Provisionaly Verified N=Not Verified S=Suspect']
Row 03: ['Site Name', '', 'Camden Kerbside', '', '', '', 'Ealing Horn Lane', '', 'Haringey Roadside', '', 'London Bexley', '']
Row 04: ['Latitude', '', '51.544233', '', '', '', '51.518950', '', '51.599300', '', '51.466030', '']
Row 05: ['Longitude', '', '-0.175228', '', '', '', '-0.265617', '', '-0.068218', '', '0.184806', '']
Row 06: ['Site Type', '', 'Urban Traffic', '', '', '', 'Urban Traffic', '', 'Urban Traffic', '', 'Suburban Background', '']
Row 07: ['Zone', '', 'Greater London', '', '', '', 'Greater London', '', 'Greater London', '', 'Greater London', '']
Row 08: ['Agglomeration', '', 'Great

## Cell 4 — Extract station names

**Why row 3, and why every second column?**  
From the inspection above we know station names are on row index 3. The columns are arranged in pairs: (PM2.5 value, Status) for each station. There are also two blank columns at the start (Date and Time). So station names appear at column positions 2, 4, 6, 8 ... — every second column starting from index 2.

In [5]:
# Row index 3 holds the station names.
# Column layout: [Date, Time, Station1_name, blank, Station2_name, blank, ...]
# So we step through columns 2, 4, 6, 8 ... (step size = 2)

station_names = []
for i in range(2, len(raw_rows[3]), 2):
    name = raw_rows[3][i].strip().strip('"')  # remove whitespace and quote marks
    if name and name.lower() != 'nan':          # skip empty or 'nan' cells
        station_names.append(name)

print(f'Found {len(station_names)} stations in this file:')
print()
for i, name in enumerate(station_names):
    print(f'  [{i}]  {name}')


Found 17 stations in this file:

  [0]  Camden Kerbside
  [1]  Ealing Horn Lane
  [2]  Haringey Roadside
  [3]  London Bexley
  [4]  London Bloomsbury
  [5]  London Eltham
  [6]  London Haringey Priory Park South
  [7]  London Harlington
  [8]  London Harrow Stanmore
  [9]  London Hillingdon
  [10]  London Honor Oak Park
  [11]  London Marylebone Road
  [12]  London N. Kensington
  [13]  London Teddington Bushy Park
  [14]  London Westminster
  [15]  Southwark A2 Old Kent Road
  [16]  Tower Hamlets Roadside


## Cell 5 — Preview one station's raw values

**Why do this?**  
Before we trust our parser, we want to see what the actual PM2.5 values look like in the raw file — including how missing values are represented (`No data`), and whether the datetime format is what we expect. This cell isolates just the first station so the output stays readable.

In [6]:
# Data starts at Excel row 12, which is Python index 11.
# Header row is Excel row 11, Python index 10.
# This preview checks how PM2.5 and Modelled Temperature appear in the raw UK-AIR file.

def parse_ukair_datetime(date_col, time_col):
    """
    Parse UK-AIR date and time columns into a proper datetime.

    The raw files may display dates differently depending on the file
    and on how Excel opens the CSV. Some dates appear in ISO style
    such as YYYY-MM-DD, while others appear in slash format such as
    M/D/YYYY.

    dayfirst=True is NOT used because the slash-format dates in these
    files are interpreted as month/day/year. Using dayfirst=True could
    misparse or drop dates such as 1/13/2021.

    The function also handles UK-AIR's 24:00:00 hour-ending notation.
    24:00:00 on date D is converted to 00:00:00 on date D+1 for
    chronological time-series modelling.
    """
    
    date_col = date_col.astype(str).str.strip()
    time_col = time_col.astype(str).str.strip()

    is_24 = time_col.str.startswith("24:")
    clean_time = time_col.copy()
    clean_time[is_24] = clean_time[is_24].str.replace("24:", "00:", regex=False)

    dt = pd.to_datetime(date_col + " " + clean_time, errors="coerce")
    dt.loc[is_24] = dt.loc[is_24] + pd.Timedelta(days=1)

    return dt


# Find PM2.5 and Modelled Temperature columns in the raw file.
target_variables = {
    "PM2.5 particulate matter (Hourly measured)": "PM25",
    "Modelled Temperature": "Temperature"
}

variable_columns = []
current_station = None

for col in range(2, len(raw_rows[10]), 2):
    station_name = raw_rows[3][col].strip() if col < len(raw_rows[3]) else ""

    if station_name:
        current_station = station_name

    raw_variable = raw_rows[10][col].strip() if col < len(raw_rows[10]) else ""

    if raw_variable in target_variables and current_station is not None:
        variable_columns.append({
            "Station": current_station,
            "Variable": target_variables[raw_variable],
            "RawVariable": raw_variable,
            "ValueCol": col,
            "StatusCol": col + 1
        })

variable_columns = pd.DataFrame(variable_columns)

print("Target variable columns found in first raw file:")
display(variable_columns.head(20))

# Choose one station that has both PM2.5 and Temperature for the preview.
stations_with_both = (
    variable_columns
    .groupby("Station")["Variable"]
    .nunique()
)

preview_station = stations_with_both[stations_with_both == 2].index[0]

preview_cols = (
    variable_columns[variable_columns["Station"] == preview_station]
    .set_index("Variable")
)

pm25_value_col = int(preview_cols.loc["PM25", "ValueCol"])
pm25_status_col = int(preview_cols.loc["PM25", "StatusCol"])

temperature_value_col = int(preview_cols.loc["Temperature", "ValueCol"])
temperature_status_col = int(preview_cols.loc["Temperature", "StatusCol"])

records = []

for row in raw_rows[11:]:
    if len(row) > max(pm25_status_col, temperature_status_col):
        records.append({
            "Date": row[0].strip(),
            "Time": row[1].strip(),
            "PM25_raw": row[pm25_value_col].strip(),
            "PM25_Status": row[pm25_status_col].strip(),
            "Temperature_raw": row[temperature_value_col].strip(),
            "Temperature_Status": row[temperature_status_col].strip()
        })

df_peek = pd.DataFrame(records)

# Keep the original reporting date for annual data capture checks.
df_peek["RawDate"] = pd.to_datetime(
    df_peek["Date"],
    errors="coerce"
)

df_peek["Year"] = df_peek["RawDate"].dt.year

# Create chronological datetime for time-series analysis.
df_peek["Datetime"] = parse_ukair_datetime(df_peek["Date"], df_peek["Time"])

# Convert PM2.5 strings to numeric values.
df_peek["PM25"] = pd.to_numeric(
    df_peek["PM25_raw"].replace(["No data", "No Data", ""], np.nan),
    errors="coerce"
)

# Convert Modelled Temperature strings to numeric values.
df_peek["Temperature"] = pd.to_numeric(
    df_peek["Temperature_raw"].replace(["No data", "No Data", ""], np.nan),
    errors="coerce"
)

# Negative PM2.5 values are physically invalid.
# This preview uses the same PM2.5 value rule as the main parser.
negative_count = (df_peek["PM25"] < 0).sum()
df_peek.loc[df_peek["PM25"] < 0, "PM25"] = np.nan

print(f"Preview station: {preview_station}")
print(f"Total rows: {len(df_peek)}")
print(f'Date range: {df_peek["Datetime"].min()}  to  {df_peek["Datetime"].max()}')
print()
print(f"Negative PM2.5 values converted to missing: {negative_count}")
print(f'Valid PM2.5 readings: {df_peek["PM25"].notna().sum()}')
print(f'Missing/invalid PM2.5 readings after preview cleaning: {df_peek["PM25"].isna().sum()}')
print()
print(f'Valid temperature readings: {df_peek["Temperature"].notna().sum()}')
print(f'Missing temperature readings: {df_peek["Temperature"].isna().sum()}')
print(f'Temperature range: {df_peek["Temperature"].min():.1f} to {df_peek["Temperature"].max():.1f} °C')
print()

df_peek[[
    "Datetime",
    "PM25_raw", "PM25_Status", "PM25",
    "Temperature_raw", "Temperature_Status", "Temperature"
]].head(15)

Target variable columns found in first raw file:


,Station,Variable,RawVariable,ValueCol,StatusCol
0,Camden Kerbside,PM25,PM2.5 particulate matter (Hourly measured),2,3
1,Camden Kerbside,Temperature,Modelled Temperature,4,5
2,Ealing Horn Lane,Temperature,Modelled Temperature,6,7
3,Haringey Roadside,Temperature,Modelled Temperature,8,9
4,London Bexley,PM25,PM2.5 particulate matter (Hourly measured),10,11
5,London Bexley,Temperature,Modelled Temperature,12,13
6,London Bloomsbury,PM25,PM2.5 particulate matter (Hourly measured),14,15
7,London Bloomsbury,Temperature,Modelled Temperature,16,17
8,London Eltham,PM25,PM2.5 particulate matter (Hourly measured),18,19
9,London Eltham,Temperature,Modelled Temperature,20,21


Preview station: Camden Kerbside
Total rows: 35064
Date range: 2017-01-01 01:00:00  to  2021-01-01 00:00:00

Negative PM2.5 values converted to missing: 215
Valid PM2.5 readings: 33188
Missing/invalid PM2.5 readings after preview cleaning: 1876

Valid temperature readings: 34368
Missing temperature readings: 696
Temperature range: -6.4 to 33.4 °C



,Datetime,PM25_raw,PM25_Status,PM25,Temperature_raw,Temperature_Status,Temperature
0,2017-01-01 01:00:00,40,V ugm-3 (TEOM FDMS),40.0,4,P degC,4.0
1,2017-01-01 02:00:00,31.6,V ugm-3 (TEOM FDMS),31.6,4.2,P degC,4.2
2,2017-01-01 03:00:00,24.7,V ugm-3 (TEOM FDMS),24.7,4.4,P degC,4.4
3,2017-01-01 04:00:00,21.2,V ugm-3 (TEOM FDMS),21.2,4.3,P degC,4.3
4,2017-01-01 05:00:00,24.9,V ugm-3 (TEOM FDMS),24.9,5.3,P degC,5.3
5,2017-01-01 06:00:00,24.6,V ugm-3 (TEOM FDMS),24.6,5.9,P degC,5.9
6,2017-01-01 07:00:00,23.9,V ugm-3 (TEOM FDMS),23.9,6.4,P degC,6.4
7,2017-01-01 08:00:00,22,V ugm-3 (TEOM FDMS),22.0,6.8,P degC,6.8
8,2017-01-01 09:00:00,19,V ugm-3 (TEOM FDMS),19.0,7.2,P degC,7.2
9,2017-01-01 10:00:00,19.9,V ugm-3 (TEOM FDMS),19.9,7.5,P degC,7.5


## Cell 6 — Raw time-axis audit and parser definition

**Raw time-axis audit before final parsing**

Before finalising the parser, the raw Date/Time sequence was checked in each UK-AIR file. This was done to confirm whether the files contain missing hourly timestamps or only non-observation rows at the end of the file.

The audit checks:

the number of valid hourly timestamps in each raw file
whether any hourly timestamps are missing
whether any duplicate timestamps exist
whether rows with invalid Date/Time values are real observations or blank/footer rows

This step is important because rows without valid Date/Time values cannot be used for annual data capture or time-series modelling. However, missing PM2.5 values at valid timestamps should be kept, because they represent the natural missingness required for the imputation study.

**Why write a function instead of repeating the code three times?**  
We have three files (2017-2020, 2021-2023, 2024-2025). Writing the same parsing logic three times would make the notebook hard to maintain — if anything needs changing, we would have to change it in three places. A single function means one change fixes everything.

**What the function does:**  
The function extracts PM2.5 and UK-AIR Modelled Temperature for each available monitoring site. PM2.5 is retained as the target pollutant variable, while Modelled Temperature is retained as an auxiliary meteorological predictor.

In [7]:
# Detailed time-axis audit for each raw UK-AIR file.
# This checks whether the raw Date/Time sequence has missing hourly timestamps
# before the main parser is finalised.
#
# This audit is about timestamps only. It does not check PM2.5 or temperature
# missing values, because those are valid data-quality/missingness issues
# analysed later.

def audit_raw_time_axis(filepath):
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        rows = list(csv.reader(f))

    target_raw_variables = [
        "PM2.5 particulate matter (Hourly measured)",
        "Modelled Temperature"
    ]

    # Count target variable columns and unique stations for the downloaded variables.
    target_variable_rows = []
    current_station = None

    for col in range(2, len(rows[10]), 2):
        station = rows[3][col].strip().strip('"') if col < len(rows[3]) else ""

        if station:
            current_station = station

        raw_variable = rows[10][col].strip() if col < len(rows[10]) else ""

        if raw_variable in target_raw_variables and current_station is not None:
            target_variable_rows.append({
                "Station": current_station,
                "RawVariable": raw_variable,
                "ValueCol": col,
                "StatusCol": col + 1
            })

    target_variable_map = pd.DataFrame(target_variable_rows)

    target_station_count = target_variable_map["Station"].nunique()
    target_variable_column_count = len(target_variable_map)

    raw_records = []

    # Actual data starts at Python row index 11.
    for raw_row_index, row in enumerate(rows[11:], start=11):
        raw_records.append({
            "RawRowIndex": raw_row_index,
            "RowLength": len(row),
            "Date": row[0].strip() if len(row) > 0 else "",
            "Time": row[1].strip() if len(row) > 1 else "",
            "FirstValue": row[0].strip() if len(row) > 0 else ""
        })

    raw_df = pd.DataFrame(raw_records)

    raw_df["RawDate"] = pd.to_datetime(raw_df["Date"], errors="coerce")
    raw_df["Datetime"] = parse_ukair_datetime(raw_df["Date"], raw_df["Time"])

    invalid_timestamp_rows = raw_df[
        raw_df["RawDate"].isna() | raw_df["Datetime"].isna()
    ].copy()

    valid_timestamp_rows = raw_df.dropna(subset=["RawDate", "Datetime"]).copy()

    actual_timestamps = pd.DatetimeIndex(
        valid_timestamp_rows["Datetime"].drop_duplicates()
    ).sort_values()

    expected_timestamps = pd.date_range(
        start=actual_timestamps.min(),
        end=actual_timestamps.max(),
        freq="h"
    )

    missing_timestamps = expected_timestamps.difference(actual_timestamps)
    duplicate_timestamps = valid_timestamp_rows["Datetime"].duplicated().sum()

    print("=" * 80)
    print(f"File: {filepath}")
    print(f"Target stations found: {target_station_count}")
    print(f"Target variable columns found: {target_variable_column_count}")
    print()
    print("Target variable summary:")
    display(
        target_variable_map
        .groupby("RawVariable")["Station"]
        .nunique()
        .reset_index(name="Number_of_stations")
    )
    print()
    print(f"Raw data rows after metadata: {len(raw_df):,}")
    print()
    print("Invalid raw timestamp rows:")
    print(f"  Raw rows with missing/invalid Date or Time: {len(invalid_timestamp_rows)}")
    print(f"  Station-time equivalent: {len(invalid_timestamp_rows)} × {target_station_count} = {len(invalid_timestamp_rows) * target_station_count}")
    print()
    print("Valid time-axis check:")
    print(f"  First valid Datetime: {actual_timestamps.min()}")
    print(f"  Last valid Datetime : {actual_timestamps.max()}")
    print(f"  Expected hourly timestamps: {len(expected_timestamps):,}")
    print(f"  Actual unique timestamps   : {len(actual_timestamps):,}")
    print(f"  Missing hourly timestamps  : {len(missing_timestamps):,}")
    print(f"  Duplicate timestamps       : {duplicate_timestamps:,}")
    print()

    print("Invalid raw rows preview:")
    display(invalid_timestamp_rows[[
        "RawRowIndex", "RowLength", "Date", "Time", "FirstValue"
    ]])

    return {
        "file": filepath,
        "target_station_count": target_station_count,
        "target_variable_column_count": target_variable_column_count,
        "invalid_raw_rows": len(invalid_timestamp_rows),
        "invalid_station_time_rows": len(invalid_timestamp_rows) * target_station_count,
        "expected_timestamps": len(expected_timestamps),
        "actual_timestamps": len(actual_timestamps),
        "missing_timestamps": len(missing_timestamps),
        "duplicate_timestamps": duplicate_timestamps
    }


time_audit_results = []

for file in FILES:
    result = audit_raw_time_axis(file)
    time_audit_results.append(result)

time_audit_summary = pd.DataFrame(time_audit_results)
time_audit_summary

File: ../data/raw/Greater_London/2017-2020.csv
Target stations found: 17
Target variable columns found: 28

Target variable summary:


,RawVariable,Number_of_stations
0,Modelled Temperature,17
1,PM2.5 particulate matter (Hourly measured),11



Raw data rows after metadata: 35,068

Invalid raw timestamp rows:
  Raw rows with missing/invalid Date or Time: 4
  Station-time equivalent: 4 × 17 = 68

Valid time-axis check:
  First valid Datetime: 2017-01-01 01:00:00
  Last valid Datetime : 2021-01-01 00:00:00
  Expected hourly timestamps: 35,064
  Actual unique timestamps   : 35,064
  Missing hourly timestamps  : 0
  Duplicate timestamps       : 0

Invalid raw rows preview:


,RawRowIndex,RowLength,Date,Time,FirstValue
35064,35075,0,,,
35065,35076,0,,,
35066,35077,0,,,
35067,35078,1,End,,End


File: ../data/raw/Greater_London/2021-2023.csv
Target stations found: 16
Target variable columns found: 27

Target variable summary:


...

File: ../data/raw/Greater_London/2024-2025.csv
Target stations found: 17
Target variable columns found: 30

Target variable summary:


,RawVariable,Number_of_stations
0,Modelled Temperature,17
1,PM2.5 particulate matter (Hourly measured),13



Raw data rows after metadata: 17,548

Invalid raw timestamp rows:
  Raw rows with missing/invalid Date or Time: 4
  Station-time equivalent: 4 × 17 = 68

Valid time-axis check:
  First valid Datetime: 2024-01-01 01:00:00
  Last valid Datetime : 2026-01-01 00:00:00
  Expected hourly timestamps: 17,544
  Actual unique timestamps   : 17,544
  Missing hourly timestamps  : 0
  Duplicate timestamps       : 0

Invalid raw rows preview:


,RawRowIndex,RowLength,Date,Time,FirstValue
17544,17555,0,,,
17545,17556,0,,,
17546,17557,0,,,
17547,17558,1,End,,End


,file,target_station_count,target_variable_column_count,invalid_raw_rows,invalid_station_time_rows,expected_timestamps,actual_timestamps,missing_timestamps,duplicate_timestamps
0,../data/raw/Greater_London/2017-2020.csv,17,28,4,68,35064,35064,0,0
1,../data/raw/Greater_London/2021-2023.csv,16,27,4,64,26280,26280,0,0
2,../data/raw/Greater_London/2024-2025.csv,17,30,4,68,17544,17544,0,0


In [8]:
TARGET_VARIABLES = {
    "PM2.5 particulate matter (Hourly measured)": "PM25",
    "Modelled Temperature": "Temperature"
}

MISSING_STRINGS = ["No data", "No Data", "no data", "", "NA", "NaN"]


def build_variable_map(filepath):
    """
    Build a map showing which raw UK-AIR columns contain PM2.5 and
    Modelled Temperature for each station.

    UK-AIR files contain value/status column pairs. When multiple variables
    are downloaded together, station metadata may appear once and then apply
    to following variable columns. Therefore, station information is
    forward-filled across measurement columns.
    """
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        rows = list(csv.reader(f))

    variable_map = []
    current_station = None
    current_metadata = {}

    
    for col in range(2, len(rows[10])):
        station_name = rows[3][col].strip().strip('"') if col < len(rows[3]) else ""

        if station_name:
            current_station = station_name
            current_metadata = {
                "Latitude": rows[4][col].strip() if col < len(rows[4]) else "",
                "Longitude": rows[5][col].strip() if col < len(rows[5]) else "",
                "SiteType": rows[6][col].strip() if col < len(rows[6]) else "",
                "Zone": rows[7][col].strip() if col < len(rows[7]) else "",
                "Agglomeration": rows[8][col].strip() if col < len(rows[8]) else "",
                "LocalAuthority": rows[9][col].strip() if col < len(rows[9]) else ""
            }

        raw_variable = rows[10][col].strip().strip('"') if col < len(rows[10]) else ""

        if raw_variable in TARGET_VARIABLES and current_station is not None:
            variable_map.append({
                "value_col": col,
                "status_col": col + 1,
                "Station": current_station,
                "RawVariable": raw_variable,
                "Variable": TARGET_VARIABLES[raw_variable],
                **current_metadata
            })

    variable_map = pd.DataFrame(variable_map)

    if variable_map.empty:
        raise ValueError(f"No target variables found in file: {filepath}")

    duplicates = variable_map.duplicated(subset=["Station", "Variable"]).sum()

    if duplicates > 0:
        print(f"Warning: {duplicates} duplicate station-variable mappings found. Keeping first occurrence.")

    variable_map = variable_map.drop_duplicates(
        subset=["Station", "Variable"],
        keep="first"
    ).reset_index(drop=True)

    return variable_map


def parse_ukair_csv(filepath):
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        rows = list(csv.reader(f))

    variable_map = build_variable_map(filepath)

    found_variables = set(variable_map["Variable"].unique())

    assert "PM25" in found_variables, f"PM2.5 variable not found in file: {filepath}"
    assert "Temperature" in found_variables, f"Modelled Temperature variable not found in file: {filepath}"

    records = []

    for _, info in variable_map.iterrows():
        value_col = int(info["value_col"])
        status_col = int(info["status_col"])

        for row in rows[11:]:
            if len(row) <= value_col:
                continue

            records.append({
                "Date": row[0].strip() if len(row) > 0 else "",
                "Time": row[1].strip() if len(row) > 1 else "",
                "Station": info["Station"],
                "Variable": info["Variable"],
                "RawValue": row[value_col].strip(),
                "Status": row[status_col].strip() if len(row) > status_col else ""
            })

    long = pd.DataFrame(records)

    long["Datetime"] = parse_ukair_datetime(long["Date"], long["Time"])

    long["RawYear"] = pd.to_datetime(
        long["Date"],
        errors="coerce"
    ).dt.year

    long = long.dropna(subset=["RawYear", "Datetime"]).copy()
    long["RawYear"] = long["RawYear"].astype(int)

    long["Value"] = pd.to_numeric(
        long["RawValue"].replace(MISSING_STRINGS, np.nan),
        errors="coerce"
    )

    index_cols = ["Station", "Date", "Time", "Datetime", "RawYear"]

    duplicate_measurements = long.duplicated(index_cols + ["Variable"]).sum()
    assert duplicate_measurements == 0, "Duplicate Station-Date-Time-Variable rows found."

    value_wide = long.pivot(
        index=index_cols,
        columns="Variable",
        values="Value"
    ).reset_index()

    status_wide = long.pivot(
        index=index_cols,
        columns="Variable",
        values="Status"
    ).reset_index()

    status_wide = status_wide.rename(columns={
        "PM25": "PM25_Status",
        "Temperature": "Temperature_Status"
    })

    tidy = value_wide.merge(status_wide, on=index_cols, how="outer")

    for col in ["PM25", "Temperature", "PM25_Status", "Temperature_Status"]:
        if col not in tidy.columns:
            tidy[col] = np.nan

    tidy.loc[tidy["PM25"] < 0, "PM25"] = np.nan

    tidy = tidy[[
        "Station", "Date", "Time", "Datetime", "RawYear",
        "PM25", "PM25_Status",
        "Temperature", "Temperature_Status"
    ]]

    tidy.columns.name = None

    return tidy


print("Parser function defined OK.")
print("This parser extracts both PM2.5 and Modelled Temperature.")

Parser function defined OK.
This parser extracts both PM2.5 and Modelled Temperature.


In [9]:
# Check that each raw UK-AIR file is parsed correctly.
# PM2.5 missing values are expected because they represent natural missing
# or invalid PM2.5 readings.
# Temperature missing values are also expected because Modelled Temperature
# can have small gaps.

for file in FILES:
    print("=" * 80)
    print(file)
    print("=" * 80)

    variable_map = build_variable_map(file)

    print("\nVariables found in this file:")
    display(
        variable_map.groupby("Variable")["Station"]
        .nunique()
        .reset_index(name="Number_of_stations")
    )

    print("\nVariable map preview:")
    display(variable_map[[
        "Station", "Variable", "RawVariable", "value_col", "status_col"
    ]].head(20))

    parsed = parse_ukair_csv(file)

    print("\nParsed data check:")
    print("Rows:", f"{len(parsed):,}")
    print("RawYear missing after parsing:", parsed["RawYear"].isna().sum())
    print("Datetime missing after parsing:", parsed["Datetime"].isna().sum())
    print("Date range:", parsed["Datetime"].min(), "to", parsed["Datetime"].max())
    print("PM2.5 missing rows:", parsed["PM25"].isna().sum())
    print("Temperature missing rows:", parsed["Temperature"].isna().sum())

    print("\nExample 24:00:00 rows:")
    example_24 = parsed[
        parsed["Time"].astype(str).str.startswith("24:")
    ][["Date", "Time", "RawYear", "Datetime"]].head().assign(
        Datetime_str=lambda x: x["Datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    display(example_24)
    print()

../data/raw/Greater_London/2017-2020.csv

Variables found in this file:


,Variable,Number_of_stations
0,PM25,11
1,Temperature,17



Variable map preview:


,Station,Variable,RawVariable,value_col,status_col
0,Camden Kerbside,PM25,PM2.5 particulate matter (Hourly measured),2,3
1,Camden Kerbside,Temperature,Modelled Temperature,4,5
2,Ealing Horn Lane,Temperature,Modelled Temperature,6,7
3,Haringey Roadside,Temperature,Modelled Temperature,8,9
4,London Bexley,PM25,PM2.5 particulate matter (Hourly measured),10,11
5,London Bexley,Temperature,Modelled Temperature,12,13
6,London Bloomsbury,PM25,PM2.5 particulate matter (Hourly measured),14,15
7,London Bloomsbury,Temperature,Modelled Temperature,16,17
8,London Eltham,PM25,PM2.5 particulate matter (Hourly measured),18,19
9,London Eltham,Temperature,Modelled Temperature,20,21



Parsed data check:
Rows: 594,672
RawYear missing after parsing: 0
Datetime missing after parsing: 0
Date range: 2017-01-01 01:00:00 to 2021-01-01 00:00:00
PM2.5 missing rows: 291417
Temperature missing rows: 59376

Example 24:00:00 rows:


...

,Variable,Number_of_stations
0,PM25,13
1,Temperature,17



Variable map preview:


,Station,Variable,RawVariable,value_col,status_col
0,Camden Kerbside,PM25,PM2.5 particulate matter (Hourly measured),2,3
1,Camden Kerbside,Temperature,Modelled Temperature,4,5
2,Ealing Horn Lane,Temperature,Modelled Temperature,6,7
3,Haringey Roadside,Temperature,Modelled Temperature,8,9
4,London Bexley,PM25,PM2.5 particulate matter (Hourly measured),10,11
5,London Bexley,Temperature,Modelled Temperature,12,13
6,London Bloomsbury,PM25,PM2.5 particulate matter (Hourly measured),14,15
7,London Bloomsbury,Temperature,Modelled Temperature,16,17
8,London Farringdon Street,PM25,PM2.5 particulate matter (Hourly measured),18,19
9,London Farringdon Street,Temperature,Modelled Temperature,20,21



Parsed data check:
Rows: 289,776
RawYear missing after parsing: 0
Datetime missing after parsing: 0
Date range: 2024-01-01 01:00:00 to 2026-01-01 00:00:00
PM2.5 missing rows: 121989
Temperature missing rows: 27528

Example 24:00:00 rows:


,Date,Time,RawYear,Datetime,Datetime_str
23,2024-01-01,24:00:00,2024,2024-01-02,2024-01-02 00:00:00
47,2024-01-02,24:00:00,2024,2024-01-03,2024-01-03 00:00:00
71,2024-01-03,24:00:00,2024,2024-01-04,2024-01-04 00:00:00
95,2024-01-04,24:00:00,2024,2024-01-05,2024-01-05 00:00:00
119,2024-01-05,24:00:00,2024,2024-01-06,2024-01-06 00:00:00


## Cell 7 — Load all three files and combine

**Why load them separately before combining?**  
Loading separately lets us verify each file individually — we can check the row counts match what we expect before merging everything into one table. If something went wrong with one file, it is much easier to spot here than after merging.

In [10]:
# Parse each file individually first so we can check each one
print('Loading files... (this may take a few seconds)')

df_2017_2020 = parse_ukair_csv(FILES[0])
df_2021_2023 = parse_ukair_csv(FILES[1])
df_2024_2025 = parse_ukair_csv(FILES[2])

print('Done. Rows per file:')
print(f'  2017-2020 : {len(df_2017_2020):>10,.0f} rows')
print(f'  2021-2023 : {len(df_2021_2023):>10,.0f} rows')
print(f'  2024-2025 : {len(df_2024_2025):>10,.0f} rows')

# Stack all three into one big table
all_data = pd.concat([df_2017_2020, df_2021_2023, df_2024_2025], ignore_index=True)

# Keep only rows within the years we care about (sanity check)

all_data = all_data[all_data['RawYear'].between(2017, 2025)]


print()
print('Combined table:')
print(f'  Total rows : {len(all_data):,.0f}')
print(f'  Date range : {all_data["Datetime"].min()}  to  {all_data["Datetime"].max()}')
print(f'  Stations   :')
for s in sorted(all_data['Station'].unique()):
    print(f'    {s}')


Loading files... (this may take a few seconds)
Done. Rows per file:
  2017-2020 :    594,672 rows
  2021-2023 :    419,472 rows
  2024-2025 :    289,776 rows

Combined table:
  Total rows : 1,303,920
  Date range : 2017-01-01 01:00:00  to  2026-01-01 00:00:00
  Stations   :
    Camden Kerbside
    Ealing Horn Lane
    Haringey Roadside
    London Bexley
    London Bloomsbury
    London Eltham
    London Farringdon Street
    London Haringey Priory Park South
    London Harlington
    London Harrow Stanmore
    London Hillingdon
    London Honor Oak Park
    London Marylebone Road
    London N. Kensington
    London Norbury Manor School
    London Teddington Bushy Park
    London Westminster
    Southwark A2 Old Kent Road
    Tower Hamlets Roadside


In [11]:
# Combined time-axis sanity check after merging all files.
# This checks the shared hourly timeline using one example station.
# Final selected stations are validated later using expected row counts.

all_data = all_data.sort_values(["Station", "Datetime"]).reset_index(drop=True)

audit_station = sorted(all_data["Station"].unique())[0]
station_times = (
    all_data[all_data["Station"] == audit_station]["Datetime"]
    .drop_duplicates()
    .sort_values()
)

combined_expected = pd.date_range(
    station_times.min(),
    station_times.max(),
    freq="h"
)

combined_missing = combined_expected.difference(station_times)
combined_dups = all_data.duplicated(subset=["Station", "Datetime"]).sum()

print(f"Combined time axis check using: {audit_station}")
print(f"  Expected slots  : {len(combined_expected):,}")
print(f"  Missing slots   : {len(combined_missing)}")
print(f"  Duplicate rows  : {combined_dups}")

assert len(combined_missing) == 0, "Gap found in combined time axis."
assert combined_dups == 0, "Duplicate Station-Datetime rows found in combined data."

Combined time axis check using: Camden Kerbside
  Expected slots  : 78,888
  Missing slots   : 0
  Duplicate rows  : 0


## Cell 8 — Compute annual data capture per station

**What is data capture?**  
In this notebook, a valid PM2.5 reading is defined as a numeric, non-missing and non-negative PM2.5 value. Values marked as "No data", blank values, unparsable values and negative PM2.5 readings are treated as missing/invalid and are not counted as valid observations. So Data capture is the percentage of hourly readings that actually contain a valid PM2.5 measurement. For example, if a station has 8760 possible hourly readings in a year but only 7884 are valid, its data capture is 90%.

**Why 8784 for some years?**  
2020 and 2024 are leap years (366 days × 24 hours = 8784 expected readings). All other years have 365 days × 24 hours = 8760. Using the wrong denominator would make leap-year capture rates look slightly lower than they actually are.

**Why 85%?**  
The 85% threshold was used as a practical minimum annual data capture criterion for selecting PM2.5 station-years. This prioritises stations with sufficiently complete observed PM2.5 records before artificial missingness is introduced for imputation experiments. Temperature was not used for station eligibility because it is an auxiliary predictor rather than the imputation target. (UK-AIR, 2025). Stations that fall below this in any study year are excluded because we cannot reliably distinguish between natural data loss and the artificial missingness we introduce in our experiments.

In [12]:
# Define expected hours for each year (leap years have one extra day)
HOURS_PER_YEAR = {
    y: (8784 if y in [2020, 2024] else 8760)
    for y in range(2017, 2026)
}

# Identify stations that actually have a PM2.5 variable in the downloaded files.
# Temperature-only stations are excluded from the PM2.5 station-selection table.
# Build raw variable map across all downloaded files
variable_maps = []

for file in FILES:
    vm = build_variable_map(file).copy()
    vm["File"] = file
    variable_maps.append(vm)

all_variable_map = pd.concat(variable_maps, ignore_index=True)

# Candidate stations are stations that have a PM2.5 variable column
# in at least one raw downloaded file.
# Annual PM2.5 capture below will decide whether they are actually selected.
pm25_candidate_stations = sorted(
    all_variable_map.loc[
        all_variable_map["Variable"] == "PM25",
        "Station"
    ].unique()
)

# Diagnostic: show how many files contain PM2.5 for each candidate station
pm25_file_coverage = (
    all_variable_map[all_variable_map["Variable"] == "PM25"]
    .groupby("Station")["File"]
    .nunique()
    .reset_index(name="PM25_File_Count")
    .sort_values(["PM25_File_Count", "Station"], ascending=[False, True])
)

print(f"PM2.5 candidate stations from raw files: {len(pm25_candidate_stations)}")
display(pm25_file_coverage)

assert len(pm25_candidate_stations) > 0, "No PM2.5 candidate stations found."

# For each PM2.5 candidate station, calculate the % of valid PM2.5 readings in each year
summary_rows = []

for station in pm25_candidate_stations:
    row = {"Station": station}
    s_data = all_data[all_data["Station"] == station]

    for year in range(2017, 2026):
        valid = s_data[s_data["RawYear"] == year]["PM25"].notna().sum()
        expected = HOURS_PER_YEAR[year]
        row[str(year)] = round((valid / expected) * 100, 1)

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index("Station")

print()
print("Annual PM2.5 data capture table computed.")
print("Temperature is not used for station eligibility.")
print()

summary

PM2.5 candidate stations from raw files: 15


,Station,PM25_File_Count
0,Camden Kerbside,3
1,London Bexley,3
2,London Bloomsbury,3
6,London Harlington,3
9,London Honor Oak Park,3
10,London Marylebone Road,3
11,London N. Kensington,3
13,London Teddington Bushy Park,3
14,London Westminster,3
3,London Eltham,2



Annual PM2.5 data capture table computed.
Temperature is not used for station eligibility.



,2017,2018,2019,2020,2021,2022,2023,2024,2025
Station,,,,,,,,,
Camden Kerbside,97.1,87.0,96.9,97.5,63.7,72.1,95.1,95.8,97.7
London Bexley,93.5,98.7,81.8,88.3,99.9,99.4,99.8,99.9,98.5
London Bloomsbury,93.4,91.2,97.0,87.2,35.4,60.4,97.1,94.0,99.9
London Eltham,87.7,82.3,96.5,95.1,89.6,62.1,46.7,0.0,0.0
London Farringdon Street,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66.7
London Haringey Priory Park South,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.7
London Harlington,99.8,96.7,97.1,98.8,99.3,99.2,99.9,99.7,99.6
London Harrow Stanmore,28.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
London Hillingdon,0.0,0.0,0.0,0.0,0.0,65.5,97.0,99.7,95.6


## Cell 9 — Colour-coded capture table

This is the key decision table. Green cells meet the AURN threshold. Red cells are the reason a station gets excluded.

🟢 **Green** = ≥ 85% &nbsp;&nbsp; 🟡 **Yellow** = 70–84% &nbsp;&nbsp; 🔴 **Red** = below 70%

In [13]:
def colour_capture(val):
    # Apply background colour based on whether the cell meets the 85% threshold.
    # This makes it immediately obvious which stations fail and in which years.
    if val >= 85:
        return 'background-color: #c6efce; color: #276221'  # green — passes
    elif val >= 70:
        return 'background-color: #ffeb9c; color: #9c6500'  # yellow — borderline
    else:
        return 'background-color: #ffc7ce; color: #9c0006'  # red — fails

# Display the table with colours applied to every cell
summary.style.map(colour_capture).format('{:.1f}')

# --> Look at this table carefully before moving to Cell 10.
# --> Notice which years cause the most failures (hint: 2021).


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Station,,,,,,,,,
Camden Kerbside,97.1,87.0,96.9,97.5,63.7,72.1,95.1,95.8,97.7
London Bexley,93.5,98.7,81.8,88.3,99.9,99.4,99.8,99.9,98.5
London Bloomsbury,93.4,91.2,97.0,87.2,35.4,60.4,97.1,94.0,99.9
London Eltham,87.7,82.3,96.5,95.1,89.6,62.1,46.7,0.0,0.0
London Farringdon Street,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66.7
London Haringey Priory Park South,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.7
London Harlington,99.8,96.7,97.1,98.8,99.3,99.2,99.9,99.7,99.6
London Harrow Stanmore,28.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
London Hillingdon,0.0,0.0,0.0,0.0,0.0,65.5,97.0,99.7,95.6


## Cell 10 — Find the best 4-year and 5-year consecutive study periods

**Why do this programmatically?**  
Instead of choosing the study period manually, the code tests every possible consecutive 4-year and 5-year window between 2017 and 2025. For each candidate period, it counts how many stations meet the 85% annual PM2.5 data capture threshold in every year of that period. This makes the station and period selection evidence-based rather than subjective.

**Why test both 4-year and 5-year windows?**  
A 5-year period is preferred because it provides more hourly observations, captures multiple seasonal cycles, and supports a clear train, validation, and test split. However, 4-year windows are also tested to check whether a slightly shorter period would retain more monitoring stations. This allows the final decision to balance temporal coverage, station coverage, and data completeness.

**Selection rule:**
The preferred dataset should retain a long consecutive period while keeping enough stations with reliable annual data capture. For the main dissertation dataset, the 85% threshold is used to prioritise data quality before applying artificial masking and imputation experiments.

In [14]:
THRESHOLD = 85.0
all_years = list(range(2017, 2026))

def evaluate_windows(window_size):
    """
    Test every consecutive study period of a given length.
    A station passes only if it meets the data capture threshold
    in every year of that period.
    """
    results = []

    for start in range(len(all_years) - window_size + 1):
        window = all_years[start : start + window_size]

        passing = []
        for station in summary.index:
            captures = [summary.loc[station, str(y)] for y in window]
            if all(c >= THRESHOLD for c in captures):
                passing.append(station)

        results.append({
            'Period': f'{window[0]}-{window[-1]}',
            'Years': window,
            'N_passing': len(passing),
            'Stations': passing
        })

    return sorted(results, key=lambda x: (-x['N_passing'], x['Period']))


for window_size in [4, 5]:
    print(f'\nBest {window_size}-year consecutive periods:')
    print('-' * 60)

    window_results = evaluate_windows(window_size)

    for r in window_results:
        print(f"{r['Period']}  -->  {r['N_passing']} station(s) pass")
        for s in r['Stations']:
            print(f"                  {s}")
        print()


Best 4-year consecutive periods:
------------------------------------------------------------
2021-2024  -->  5 station(s) pass
                  London Bexley
                  London Harlington
                  London Honor Oak Park
                  London Marylebone Road
                  London N. Kensington

2022-2025  -->  5 station(s) pass
                  London Bexley
                  London Harlington
                  London Honor Oak Park
                  London Marylebone Road
                  London N. Kensington

2017-2020  -->  4 station(s) pass
                  Camden Kerbside
                  London Bloomsbury
                  London Harlington
                  London N. Kensington

2019-2022  -->  4 station(s) pass
                  London Harlington
                  London Honor Oak Park
                  London N. Kensington
                  London Teddington Bushy Park

2020-2023  -->  4 station(s) pass
                  London Bexley
                

## Cell 11 — Apply the final selection

**What to do:**  
Look at the Cell 10 output and set `STUDY_YEARS` to the best window. Then run this cell to see the final list of selected and excluded stations with the exact reason each excluded station failed.

In [15]:
# Final study period selected after comparing all consecutive
# 4-year and 5-year windows in Cell 10.
# 2021-2025 was chosen because it is the longest consecutive
# complete-calendar-year period retaining 5 stations with
# >=85% annual PM2.5 data capture in every year.
STUDY_YEARS = [2021, 2022, 2023, 2024, 2025]
selected = []  # stations that pass in every study year
excluded = []  # stations that fail in at least one study year

for station in summary.index:
    captures = [summary.loc[station, str(y)] for y in STUDY_YEARS]
    if all(c >= THRESHOLD for c in captures):
        selected.append(station)
    else:
        excluded.append(station)

# --- Print selected stations ---
print(f'Study period : {STUDY_YEARS[0]}-{STUDY_YEARS[-1]}')
print(f'Threshold    : >= {THRESHOLD}% in every year')
print()
print(f'SELECTED  ({len(selected)} stations):')
for s in selected:
    caps = '  |  '.join([f"{y}: {summary.loc[s, str(y)]}%" for y in STUDY_YEARS])
    print(f'  OK  {s:<40} {caps}')

# --- Print excluded stations with the specific years that caused failure ---
print()
print(f'EXCLUDED  ({len(excluded)} stations):')
for s in excluded:
    failed = [
        f"{y}: {summary.loc[s, str(y)]}%"
        for y in STUDY_YEARS
        if summary.loc[s, str(y)] < THRESHOLD
    ]
    print(f'  --  {s:<40} failed in: {"  |  ".join(failed)}')

# This output is what you report in your dissertation methodology section.
# Example sentence:
# 'X stations were excluded because their annual data capture fell below the
# 85% AURN threshold in at least one study year (UK-AIR, 2025).'


Study period : 2021-2025
Threshold    : >= 85.0% in every year

SELECTED  (5 stations):
  OK  London Bexley                            2021: 99.9%  |  2022: 99.4%  |  2023: 99.8%  |  2024: 99.9%  |  2025: 98.5%
  OK  London Harlington                        2021: 99.3%  |  2022: 99.2%  |  2023: 99.9%  |  2024: 99.7%  |  2025: 99.6%
  OK  London Honor Oak Park                    2021: 96.7%  |  2022: 99.2%  |  2023: 99.6%  |  2024: 99.6%  |  2025: 98.7%
  OK  London Marylebone Road                   2021: 85.4%  |  2022: 85.3%  |  2023: 94.0%  |  2024: 94.4%  |  2025: 96.2%
  OK  London N. Kensington                     2021: 99.9%  |  2022: 98.4%  |  2023: 99.6%  |  2024: 99.6%  |  2025: 99.8%

EXCLUDED  (10 stations):
  --  Camden Kerbside                          failed in: 2021: 63.7%  |  2022: 72.1%
  --  London Bloomsbury                        failed in: 2021: 35.4%  |  2022: 60.4%
  --  London Eltham                            failed in: 2022: 62.1%  |  2023: 46.7%  |  2024: 0.0

## Cell 12 — Save the final dataset

**Why save to `data/processed/` and not overwrite the raw files?**  
The raw files are the original download from UK-AIR and should never be modified. Saving the processed output to a separate folder means we can always go back to the raw data if something needs to be redone. This is standard practice in reproducible data science.

In [16]:
# This long-format panel is used for EDA, masking and model evaluation.
# Station selection is based on PM2.5 data capture.
# Temperature is retained as an auxiliary meteorological predictor.

final = all_data[
    (all_data["Station"].isin(selected)) &
    (all_data["RawYear"].isin(STUDY_YEARS))
].copy()

# Keep PM2.5, temperature, and status columns for audit/EDA.
final = final[[
    "Station", "Datetime", "RawYear",
    "PM25", "PM25_Status",
    "Temperature", "Temperature_Status"
]]

# Remove any leftover column-index name from pivot operations.
final.columns.name = None

final = final.sort_values(["Station", "Datetime"]).reset_index(drop=True)
final["RawYear"] = final["RawYear"].astype(int)

expected_rows = sum(HOURS_PER_YEAR[y] for y in STUDY_YEARS) * len(selected)
actual_rows = len(final)

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {actual_rows:,}")

assert actual_rows == expected_rows, "Row count does not match expected station-hour total."

duplicate_count = final.duplicated(subset=["Station", "Datetime"]).sum()
print(f"Duplicate station-datetime rows: {duplicate_count}")

assert duplicate_count == 0, "Duplicate station-datetime rows found."

rows_per_station = final.groupby("Station").size()
print("\nRows per station:")
print(rows_per_station)

assert rows_per_station.nunique() == 1, "Stations do not have equal hourly row counts."

# Missingness flags for EDA only.
# These will not be used as model features.
final["PM25_Missing"] = final["PM25"].isna()
final["Temperature_Missing"] = final["Temperature"].isna()

final["Both_PM25_and_Temperature_Missing"] = (
    final["PM25_Missing"] & final["Temperature_Missing"]
)

final["PM25_Missing_Temperature_Available"] = (
    final["PM25_Missing"] & ~final["Temperature_Missing"]
)

final["PM25_Available_Temperature_Missing"] = (
    ~final["PM25_Missing"] & final["Temperature_Missing"]
)

print("\nFinal panel validation passed.")
print("Missingness flags added for EDA only.")
display(final.head())

Expected rows: 219,120
Actual rows:   219,120
Duplicate station-datetime rows: 0

Rows per station:
Station
London Bexley             43824
London Harlington         43824
London Honor Oak Park     43824
London Marylebone Road    43824
London N. Kensington      43824
dtype: int64

Final panel validation passed.
Missingness flags added for EDA only.


,Station,Datetime,RawYear,PM25,PM25_Status,Temperature,Temperature_Status,PM25_Missing,Temperature_Missing,Both_PM25_and_Temperature_Missing,PM25_Missing_Temperature_Available,PM25_Available_Temperature_Missing
0,London Bexley,2021-01-01 01:00:00,2021,39.552,V ugm-3 (Ref.eq),-1.2,N degC,False,False,False,False,False
1,London Bexley,2021-01-01 02:00:00,2021,52.359,V ugm-3 (Ref.eq),-1.9,N degC,False,False,False,False,False
2,London Bexley,2021-01-01 03:00:00,2021,20.920,V ugm-3 (Ref.eq),-2.3,N degC,False,False,False,False,False
3,London Bexley,2021-01-01 04:00:00,2021,15.943,V ugm-3 (Ref.eq),-2.8,N degC,False,False,False,False,False
4,London Bexley,2021-01-01 05:00:00,2021,17.076,V ugm-3 (Ref.eq),-3.2,N degC,False,False,False,False,False


In [17]:
# Status flag audit.
# PM2.5 is a measured pollutant variable.
# Temperature is a modelled meteorological variable, so its status codes
# are interpreted differently from measured pollutant verification flags.

pm25_status_audit = (
    final["PM25_Status"]
    .value_counts(dropna=False)
    .rename_axis("PM25_Status")
    .reset_index(name="Count")
)

temperature_status_audit = (
    final["Temperature_Status"]
    .value_counts(dropna=False)
    .rename_axis("Temperature_Status")
    .reset_index(name="Count")
)

print("PM2.5 status values:")
display(pm25_status_audit)

print("Temperature status values:")
display(temperature_status_audit)

PM2.5 status values:


,PM25_Status,Count
0,V ugm-3 (Ref.eq),175166
1,V ugm-3 (BAM),35293
2,V ugm-3 (TEOM FDMS),8531
3,,130


Temperature status values:


,Temperature_Status,Count
0,N degC,214920
1,,4200


In [18]:
# ============================================================
# Additional audit: PM2.5 status codes by station and year
# ============================================================

pm25_status_by_station_year = (
    final
    .groupby(["Station", "RawYear", "PM25_Status"], dropna=False)
    .size()
    .reset_index(name="Rows")
    .sort_values(["Station", "RawYear", "PM25_Status"])
)

display(pm25_status_by_station_year)

marylebone_status_audit = (
    pm25_status_by_station_year[
        pm25_status_by_station_year["Station"] == "London Marylebone Road"
    ]
    .copy()
)

print("Marylebone Road PM2.5 status audit:")
display(marylebone_status_audit)

,Station,RawYear,PM25_Status,Rows
0,London Bexley,2021,V ugm-3 (Ref.eq),8760
1,London Bexley,2022,V ugm-3 (Ref.eq),8760
2,London Bexley,2023,V ugm-3 (Ref.eq),8760
3,London Bexley,2024,V ugm-3 (Ref.eq),8784
4,London Bexley,2025,V ugm-3 (Ref.eq),8760
5,London Harlington,2021,,47
6,London Harlington,2021,V ugm-3 (Ref.eq),8713
7,London Harlington,2022,,58
8,London Harlington,2022,V ugm-3 (Ref.eq),8702
9,London Harlington,2023,,5


Marylebone Road PM2.5 status audit:


,Station,RawYear,PM25_Status,Rows
20,London Marylebone Road,2021,V ugm-3 (BAM),229
21,London Marylebone Road,2021,V ugm-3 (TEOM FDMS),8531
22,London Marylebone Road,2022,V ugm-3 (BAM),8760
23,London Marylebone Road,2023,V ugm-3 (BAM),8760
24,London Marylebone Road,2024,V ugm-3 (BAM),8784
25,London Marylebone Road,2025,V ugm-3 (BAM),8760


In [19]:
def missingness_report(df, name="DataFrame"):
    """
    Report missing values for every column.
    For object columns, blank strings and 'No data' are also treated as missing.
    """
    rows = []

    for col in df.columns:
        series = df[col]

        if series.dtype == "object":
            missing = (
                series.isna() |
                series.astype(str).str.strip().isin(["", "No data", "No Data", "no data"])
            )
        else:
            missing = series.isna()

        rows.append({
            "Column": col,
            "MissingCount": int(missing.sum()),
            "Missing_%": round(missing.mean() * 100, 2),
            "NonMissingCount": int((~missing).sum()),
            "NonMissing_%": round((~missing).mean() * 100, 2),
            "Dtype": str(series.dtype)
        })

    report = pd.DataFrame(rows).sort_values("Missing_%", ascending=False)

    print(f"Missingness report for: {name}")
    print(f"Rows: {len(df):,}")
    display(report)

    return report


final_missingness_report = missingness_report(
    final,
    "Final selected PM2.5 + temperature panel"
)

Missingness report for: Final selected PM2.5 + temperature panel
Rows: 219,120


,Column,MissingCount,Missing_%,NonMissingCount,NonMissing_%,Dtype
3,PM25,5107,2.33,214013,97.67,float64
5,Temperature,4200,1.92,214920,98.08,float64
6,Temperature_Status,4200,1.92,214920,98.08,object
4,PM25_Status,130,0.06,218990,99.94,object
0,Station,0,0.00,219120,100.00,object
1,Datetime,0,0.00,219120,100.00,datetime64[ns]
2,RawYear,0,0.00,219120,100.00,int32
7,PM25_Missing,0,0.00,219120,100.00,bool
8,Temperature_Missing,0,0.00,219120,100.00,bool
9,Both_PM25_and_Temperature_Missing,0,0.00,219120,100.00,bool


In [20]:
overlap_summary = pd.DataFrame({
    "Category": [
        "Both PM2.5 and Temperature available",
        "PM2.5 missing, Temperature available",
        "PM2.5 available, Temperature missing",
        "Both PM2.5 and Temperature missing"
    ],
    "Rows": [
        ((~final["PM25_Missing"]) & (~final["Temperature_Missing"])).sum(),
        ((final["PM25_Missing"]) & (~final["Temperature_Missing"])).sum(),
        ((~final["PM25_Missing"]) & (final["Temperature_Missing"])).sum(),
        ((final["PM25_Missing"]) & (final["Temperature_Missing"])).sum()
    ]
})

overlap_summary["Percentage"] = round(
    overlap_summary["Rows"] / len(final) * 100,
    2
)

print("Overlap between PM2.5 and temperature missingness:")
display(overlap_summary)

Overlap between PM2.5 and temperature missingness:


,Category,Rows,Percentage
0,Both PM2.5 and Temperature available,209918,95.80
1,"PM2.5 missing, Temperature available",5002,2.28
2,"PM2.5 available, Temperature missing",4095,1.87
3,Both PM2.5 and Temperature missing,105,0.05


In [21]:
station_year_missingness = (
    final
    .groupby(["Station", "RawYear"])
    .agg(
        Rows=("PM25", "size"),
        PM25_Valid=("PM25", lambda x: x.notna().sum()),
        PM25_Missing=("PM25_Missing", "sum"),
        Temperature_Valid=("Temperature", lambda x: x.notna().sum()),
        Temperature_Missing=("Temperature_Missing", "sum"),
        Both_Missing=("Both_PM25_and_Temperature_Missing", "sum"),
        PM25_Missing_Temperature_Available=("PM25_Missing_Temperature_Available", "sum"),
        PM25_Available_Temperature_Missing=("PM25_Available_Temperature_Missing", "sum")
    )
    .reset_index()
)

station_year_missingness["PM25_Capture_%"] = round(
    station_year_missingness["PM25_Valid"] / station_year_missingness["Rows"] * 100,
    2
)

station_year_missingness["Temperature_Capture_%"] = round(
    station_year_missingness["Temperature_Valid"] / station_year_missingness["Rows"] * 100,
    2
)

station_year_missingness["PM25_Missing_%"] = round(
    station_year_missingness["PM25_Missing"] / station_year_missingness["Rows"] * 100,
    2
)

station_year_missingness["Temperature_Missing_%"] = round(
    station_year_missingness["Temperature_Missing"] / station_year_missingness["Rows"] * 100,
    2
)

station_year_missingness["Both_Missing_%"] = round(
    station_year_missingness["Both_Missing"] / station_year_missingness["Rows"] * 100,
    2
)

print("Station-year missingness and availability summary:")
display(station_year_missingness)

Station-year missingness and availability summary:


,Station,RawYear,Rows,PM25_Valid,PM25_Missing,Temperature_Valid,Temperature_Missing,Both_Missing,PM25_Missing_Temperature_Available,PM25_Available_Temperature_Missing,PM25_Capture_%,Temperature_Capture_%,PM25_Missing_%,Temperature_Missing_%,Both_Missing_%
0,London Bexley,2021,8760,8754,6,8400,360,0,6,360,99.93,95.89,0.07,4.11,0.00
1,London Bexley,2022,8760,8711,49,8664,96,0,49,96,99.44,98.90,0.56,1.10,0.00
2,London Bexley,2023,8760,8740,20,8712,48,0,20,48,99.77,99.45,0.23,0.55,0.00
3,London Bexley,2024,8784,8772,12,8664,120,0,12,120,99.86,98.63,0.14,1.37,0.00
4,London Bexley,2025,8760,8630,130,8544,216,0,130,216,98.52,97.53,1.48,2.47,0.00
5,London Harlington,2021,8760,8703,57,8400,360,0,57,360,99.35,95.89,0.65,4.11,0.00
6,London Harlington,2022,8760,8694,66,8664,96,0,66,96,99.25,98.90,0.75,1.10,0.00
7,London Harlington,2023,8760,8749,11,8712,48,0,11,48,99.87,99.45,0.13,0.55,0.00
8,London Harlington,2024,8784,8756,28,8664,120,0,28,120,99.68,98.63,0.32,1.37,0.00
9,London Harlington,2025,8760,8728,32,8544,216,0,32,216,99.63,97.53,0.37,2.47,0.00


In [22]:
# Temperature availability and plausibility audit.
# Temperature is not used for the 85% station-selection threshold.
# It is checked separately because it is an auxiliary meteorological predictor.

temperature_audit = (
    final
    .groupby(["Station", "RawYear"])
    .agg(
        Rows=("Temperature", "size"),
        Temperature_Valid=("Temperature", lambda x: x.notna().sum()),
        Temperature_Missing=("Temperature", lambda x: x.isna().sum()),
        Temperature_Min=("Temperature", "min"),
        Temperature_Max=("Temperature", "max"),
        Temperature_Mean=("Temperature", "mean"),
        PM25_Valid=("PM25", lambda x: x.notna().sum()),
        PM25_Missing=("PM25", lambda x: x.isna().sum())
    )
    .reset_index()
)

temperature_audit["ExpectedHours"] = temperature_audit["RawYear"].map(HOURS_PER_YEAR)

temperature_audit["Temperature_Capture_%"] = round(
    temperature_audit["Temperature_Valid"] / temperature_audit["ExpectedHours"] * 100,
    2
)

temperature_audit["PM25_Capture_%"] = round(
    temperature_audit["PM25_Valid"] / temperature_audit["ExpectedHours"] * 100,
    2
)

temperature_audit["Temperature_Min"] = round(temperature_audit["Temperature_Min"], 1)
temperature_audit["Temperature_Max"] = round(temperature_audit["Temperature_Max"], 1)
temperature_audit["Temperature_Mean"] = round(temperature_audit["Temperature_Mean"], 2)

print("Temperature availability and plausibility audit:")
display(temperature_audit)

overall_temperature_capture = final["Temperature"].notna().mean() * 100
overall_pm25_capture = final["PM25"].notna().mean() * 100

print("Overall final-panel availability:")
print(f"PM2.5 capture:       {overall_pm25_capture:.2f}%")
print(f"Temperature capture: {overall_temperature_capture:.2f}%")

print()
print("Temperature range in final panel:")
print(f"Minimum temperature: {final['Temperature'].min():.1f} °C")
print(f"Maximum temperature: {final['Temperature'].max():.1f} °C")


min_temperature_capture = temperature_audit["Temperature_Capture_%"].min()

print(f"Minimum station-year temperature capture: {min_temperature_capture:.2f}%")

assert min_temperature_capture >= 85, (
    "At least one selected station-year has temperature capture below 85%. "
    "Temperature should be reviewed before using it as an auxiliary model predictor."
)

Temperature availability and plausibility audit:


,Station,RawYear,Rows,Temperature_Valid,Temperature_Missing,Temperature_Min,Temperature_Max,Temperature_Mean,PM25_Valid,PM25_Missing,ExpectedHours,Temperature_Capture_%,PM25_Capture_%
0,London Bexley,2021,8760,8400,360,-5.2,27.4,9.42,8754,6,8760,95.89,99.93
1,London Bexley,2022,8760,8664,96,-5.3,29.9,10.62,8711,49,8760,98.90,99.44
2,London Bexley,2023,8760,8712,48,-4.5,28.0,10.52,8740,20,8760,99.45,99.77
3,London Bexley,2024,8784,8664,120,-5.7,28.4,10.57,8772,12,8784,98.63,99.86
4,London Bexley,2025,8760,8544,216,-4.8,29.1,10.38,8630,130,8760,97.53,98.52
5,London Harlington,2021,8760,8400,360,-5.8,27.7,9.43,8703,57,8760,95.89,99.35
6,London Harlington,2022,8760,8664,96,-5.5,29.7,10.65,8694,66,8760,98.90,99.25
7,London Harlington,2023,8760,8712,48,-5.5,29.0,10.52,8749,11,8760,99.45,99.87
8,London Harlington,2024,8784,8664,120,-6.1,27.5,10.48,8756,28,8784,98.63,99.68
9,London Harlington,2025,8760,8544,216,-4.6,27.7,10.47,8728,32,8760,97.53,99.63


Overall final-panel availability:
PM2.5 capture:       97.67%
Temperature capture: 98.08%

Temperature range in final panel:
Minimum temperature: -7.5 °C
Maximum temperature: 31.9 °C
Minimum station-year temperature capture: 95.89%


In [23]:
# Save clean modelling base panel.
# This file does not include missingness flags, because flags are for EDA only.
model_base = final[[
    "Station", "Datetime", "RawYear",
    "PM25", "Temperature"
]].copy()

model_base_output_path = "../data/processed/selected_pm25_temperature_panel.csv"
model_base.to_csv(model_base_output_path, index=False)

print(f"Saved modelling base panel to: {model_base_output_path}")
print(f"Rows: {len(model_base):,}")
print(f"Columns: {list(model_base.columns)}")
display(model_base.head())

Saved modelling base panel to: ../data/processed/selected_pm25_temperature_panel.csv
Rows: 219,120
Columns: ['Station', 'Datetime', 'RawYear', 'PM25', 'Temperature']


,Station,Datetime,RawYear,PM25,Temperature
0,London Bexley,2021-01-01 01:00:00,2021,39.552,-1.2
1,London Bexley,2021-01-01 02:00:00,2021,52.359,-1.9
2,London Bexley,2021-01-01 03:00:00,2021,20.920,-2.3
3,London Bexley,2021-01-01 04:00:00,2021,15.943,-2.8
4,London Bexley,2021-01-01 05:00:00,2021,17.076,-3.2


In [24]:
# Save EDA panel with missingness flags.
# These flags are used only for missingness analysis, not as model features.
eda_panel = final[[
    "Station", "Datetime", "RawYear",
    "PM25", "PM25_Status",
    "Temperature", "Temperature_Status",
    "PM25_Missing", "Temperature_Missing",
    "Both_PM25_and_Temperature_Missing",
    "PM25_Missing_Temperature_Available",
    "PM25_Available_Temperature_Missing"
]].copy()

eda_output_path = "../data/processed/selected_pm25_temperature_panel_for_eda.csv"
eda_panel.to_csv(eda_output_path, index=False)

print(f"Saved EDA panel to: {eda_output_path}")
print(f"Rows: {len(eda_panel):,}")
print(f"Columns: {list(eda_panel.columns)}")
display(eda_panel.head())

Saved EDA panel to: ../data/processed/selected_pm25_temperature_panel_for_eda.csv
Rows: 219,120
Columns: ['Station', 'Datetime', 'RawYear', 'PM25', 'PM25_Status', 'Temperature', 'Temperature_Status', 'PM25_Missing', 'Temperature_Missing', 'Both_PM25_and_Temperature_Missing', 'PM25_Missing_Temperature_Available', 'PM25_Available_Temperature_Missing']


,Station,Datetime,RawYear,PM25,PM25_Status,Temperature,Temperature_Status,PM25_Missing,Temperature_Missing,Both_PM25_and_Temperature_Missing,PM25_Missing_Temperature_Available,PM25_Available_Temperature_Missing
0,London Bexley,2021-01-01 01:00:00,2021,39.552,V ugm-3 (Ref.eq),-1.2,N degC,False,False,False,False,False
1,London Bexley,2021-01-01 02:00:00,2021,52.359,V ugm-3 (Ref.eq),-1.9,N degC,False,False,False,False,False
2,London Bexley,2021-01-01 03:00:00,2021,20.920,V ugm-3 (Ref.eq),-2.3,N degC,False,False,False,False,False
3,London Bexley,2021-01-01 04:00:00,2021,15.943,V ugm-3 (Ref.eq),-2.8,N degC,False,False,False,False,False
4,London Bexley,2021-01-01 05:00:00,2021,17.076,V ugm-3 (Ref.eq),-3.2,N degC,False,False,False,False,False


In [25]:
# Save missingness and temperature audit outputs for reporting and reproducibility.

missingness_report_path = "../data/processed/final_missingness_report.csv"
overlap_summary_path = "../data/processed/pm25_temperature_missingness_overlap.csv"
station_year_missingness_path = "../data/processed/station_year_missingness_summary.csv"
temperature_audit_path = "../data/processed/temperature_availability_audit.csv"
pm25_status_audit_path = "../data/processed/pm25_status_audit.csv"
temperature_status_audit_path = "../data/processed/temperature_status_audit.csv"
pm25_status_by_station_year_path = "../data/processed/pm25_status_by_station_year.csv"
marylebone_status_audit_path = "../data/processed/marylebone_pm25_status_audit.csv"


pm25_status_audit.to_csv(pm25_status_audit_path, index=False)
temperature_status_audit.to_csv(temperature_status_audit_path, index=False)

final_missingness_report.to_csv(missingness_report_path, index=False)
overlap_summary.to_csv(overlap_summary_path, index=False)
station_year_missingness.to_csv(station_year_missingness_path, index=False)
temperature_audit.to_csv(temperature_audit_path, index=False)

pm25_status_by_station_year.to_csv(pm25_status_by_station_year_path, index=False)
marylebone_status_audit.to_csv(marylebone_status_audit_path, index=False)

print("Saved audit files:")
print(f"  {missingness_report_path}")
print(f"  {overlap_summary_path}")
print(f"  {station_year_missingness_path}")
print(f"  {temperature_audit_path}")
print(f"  {pm25_status_audit_path}")
print(f"  {temperature_status_audit_path}")
print(f"  {pm25_status_by_station_year_path}")
print(f"  {marylebone_status_audit_path}")

Saved audit files:
  ../data/processed/final_missingness_report.csv
  ../data/processed/pm25_temperature_missingness_overlap.csv
  ../data/processed/station_year_missingness_summary.csv
  ../data/processed/temperature_availability_audit.csv
  ../data/processed/pm25_status_audit.csv
  ../data/processed/temperature_status_audit.csv
  ../data/processed/pm25_status_by_station_year.csv
  ../data/processed/marylebone_pm25_status_audit.csv


In [26]:
# Save wide PM2.5 panel.
# This is needed for EDA, station correlations, masking design,
# and later spatial feature engineering.

# Save wide PM2.5 panel.
wide_pm25 = model_base.pivot(
    index="Datetime",
    columns="Station",
    values="PM25"
).sort_index()

wide_pm25.columns.name = None

expected_timestamps = sum(HOURS_PER_YEAR[y] for y in STUDY_YEARS)
expected_stations = len(selected)

print(f"Wide PM2.5 panel shape: {wide_pm25.shape}")

assert wide_pm25.shape == (expected_timestamps, expected_stations), (
    "Wide PM2.5 shape is not as expected."
)

expected_index = pd.date_range(
    start=wide_pm25.index.min(),
    end=wide_pm25.index.max(),
    freq="h"
)

assert wide_pm25.index.equals(expected_index), (
    "Wide PM2.5 panel index is not a complete hourly DatetimeIndex."
)

pm25_wide_output_path = "../data/processed/selected_pm25_wide_panel.csv"
wide_pm25.to_csv(pm25_wide_output_path)

print(f"Saved wide PM2.5 panel to: {pm25_wide_output_path}")
display(wide_pm25.head())

Wide PM2.5 panel shape: (43824, 5)
Saved wide PM2.5 panel to: ../data/processed/selected_pm25_wide_panel.csv


,London Bexley,London Harlington,London Honor Oak Park,London Marylebone Road,London N. Kensington
Datetime,,,,,
2021-01-01 01:00:00,39.552,22.453,29.316,18.6,30.448
2021-01-01 02:00:00,52.359,19.099,19.293,40.3,55.802
2021-01-01 03:00:00,20.920,20.925,16.604,23.9,28.278
2021-01-01 04:00:00,15.943,19.974,15.330,18.1,15.613
2021-01-01 05:00:00,17.076,17.363,17.807,21.7,19.764


In [27]:
# Save wide temperature panel.
# This is useful for EDA and for checking whether modelled temperature
# behaves consistently across the selected AURN sites.

# Save wide temperature panel.
wide_temperature = model_base.pivot(
    index="Datetime",
    columns="Station",
    values="Temperature"
).sort_index()

wide_temperature.columns.name = None

expected_timestamps = sum(HOURS_PER_YEAR[y] for y in STUDY_YEARS)
expected_stations = len(selected)

print(f"Wide temperature panel shape: {wide_temperature.shape}")

assert wide_temperature.shape == (expected_timestamps, expected_stations), (
    "Wide temperature shape is not as expected."
)

expected_index = pd.date_range(
    start=wide_temperature.index.min(),
    end=wide_temperature.index.max(),
    freq="h"
)

assert wide_temperature.index.equals(expected_index), (
    "Wide temperature panel index is not a complete hourly DatetimeIndex."
)

temperature_wide_output_path = "../data/processed/selected_temperature_wide_panel.csv"
wide_temperature.to_csv(temperature_wide_output_path)

print(f"Saved wide temperature panel to: {temperature_wide_output_path}")
display(wide_temperature.head())

Wide temperature panel shape: (43824, 5)
Saved wide temperature panel to: ../data/processed/selected_temperature_wide_panel.csv


,London Bexley,London Harlington,London Honor Oak Park,London Marylebone Road,London N. Kensington
Datetime,,,,,
2021-01-01 01:00:00,-1.2,-1.5,-2.1,-2.1,-2.1
2021-01-01 02:00:00,-1.9,-2.1,-2.9,-2.9,-2.9
2021-01-01 03:00:00,-2.3,-2.6,-3.5,-3.6,-3.6
2021-01-01 04:00:00,-2.8,-3.0,-4.0,-4.0,-4.0
2021-01-01 05:00:00,-3.2,-3.2,-4.4,-4.5,-4.5


In [28]:
# Save selected station metadata.
# Station metadata is needed later for spatial features such as nearest
# neighbours and inverse-distance weighted PM2.5 features.

def extract_station_metadata(filepath):
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        rows = list(csv.reader(f))

    metadata_rows = []

    for col in range(2, len(rows[3]), 2):
        station = rows[3][col].strip().strip('"') if col < len(rows[3]) else ""

        if not station:
            continue

        metadata_rows.append({
            "Station": station,
            "Latitude": rows[4][col].strip() if col < len(rows[4]) else "",
            "Longitude": rows[5][col].strip() if col < len(rows[5]) else "",
            "SiteType": rows[6][col].strip() if col < len(rows[6]) else "",
            "Zone": rows[7][col].strip() if col < len(rows[7]) else "",
            "Agglomeration": rows[8][col].strip() if col < len(rows[8]) else "",
            "LocalAuthority": rows[9][col].strip() if col < len(rows[9]) else ""
        })

    return pd.DataFrame(metadata_rows)


metadata_all = pd.concat(
    [extract_station_metadata(f) for f in FILES],
    ignore_index=True
)

metadata_all["Latitude"] = pd.to_numeric(metadata_all["Latitude"], errors="coerce")
metadata_all["Longitude"] = pd.to_numeric(metadata_all["Longitude"], errors="coerce")

metadata_all = metadata_all.dropna(subset=["Station"])
metadata_all = metadata_all.drop_duplicates(subset=["Station"])

selected_metadata = metadata_all[
    metadata_all["Station"].isin(selected)
].copy()

selected_metadata = selected_metadata.sort_values("Station").reset_index(drop=True)

assert len(selected_metadata) == len(selected), "Metadata missing for one or more selected stations."
assert selected_metadata["Latitude"].notna().all(), "Missing latitude found."
assert selected_metadata["Longitude"].notna().all(), "Missing longitude found."

metadata_output_path = "../data/processed/selected_station_metadata.csv"
selected_metadata.to_csv(metadata_output_path, index=False)

print(f"Saved station metadata to: {metadata_output_path}")
display(selected_metadata)

Saved station metadata to: ../data/processed/selected_station_metadata.csv


,Station,Latitude,Longitude,SiteType,Zone,Agglomeration,LocalAuthority
0,London Bexley,51.466030,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
1,London Harlington,51.488790,-0.441614,Urban Industrial,Greater London,Greater London Urban Area,Hillingdon
2,London Honor Oak Park,51.449674,-0.037418,Urban Background,Greater London,Greater London Urban Area,Lewisham
3,London Marylebone Road,51.522530,-0.154611,Urban Traffic,Greater London,Greater London Urban Area,Westminster
4,London N. Kensington,51.521050,-0.213419,Urban Background,Greater London,Greater London Urban Area,Kensington


## Final Notebook 01 Outputs

This notebook selected the final PM2.5 study period and AURN monitoring stations using an 85% annual PM2.5 data capture threshold.

The final selected dataset covers 2021–2025 and includes five London AURN stations. PM2.5 was retained as the target pollutant variable, while UK-AIR Modelled Temperature was retained as an auxiliary meteorological predictor.

The final panel contains 219,120 station-hour rows. Validation confirmed that the expected row count matches the actual row count, there are no duplicate Station-Datetime rows, and each station has the same number of hourly records.

Missingness indicators were created for EDA only and were not saved in the main modelling input file as predictive features.